### Allscripts Sunrise (SCM) Procedure Occurrence Hydration

This notebook currently derives procedures from Sunrise orders/tasks so `procedure_occurrence` is no longer a commented-out template.

Client follow-up also provided a separate SCM billing extract pattern in `(Clone) procedures_SCM.py` that sources CPT/HCPCS-style procedures from Soarian, DSS, and Athena `omny_accounts` feeds through an SCM encounter mapper. Treat that billing flow as complementary source context that still needs reconciliation with this OMOP hydration path and its concept-mapping strategy.

In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.procedure_occurrence;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.procedure_occurrence
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_procedure_occurrence
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_procedure_occurrence AS
WITH staged AS (
  SELECT
    CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3order', 'GUID', CAST(ord.GUID AS STRING)) AS procedure_occurrence_source_value,
    stp.person_id,
    COALESCE(proc_concept.omop_concept_id, 0) AS procedure_concept_id,
    CAST(COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS DATE) AS procedure_date,
    COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) AS procedure_datetime,
    32817 AS procedure_type_concept_id,
    COALESCE(mod_concept.omop_concept_id, 0) AS modifier_concept_id,
    CAST(1 AS DOUBLE) AS quantity,
    stpr.provider_id AS provider_id,
    stvo.visit_occurrence_id AS visit_occurrence_id,
    NULL AS visit_detail_id,
    COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode) AS procedure_source_value,
    0 AS procedure_source_concept_id,
    NULLIF(TRIM(ord.Modifier), '') AS modifier_source_value,
    'allscripts_scm' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
  LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
    ON oto.OrderGUID = ord.GUID
   AND oto.Active = TRUE
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(COALESCE(oto.ClientGUID, ord.ClientGUID) AS STRING))
   AND stp.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_provider stpr
    ON stpr.provider_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3careprovider', 'GUID', CAST(COALESCE(oto.PerformedProviderGUID, oto.EnteredProviderGUID, ord.CareProviderGUID) AS STRING))
   AND stpr.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT('allscripts_scm', ' | ', CAST(ord.ClientVisitGUID AS STRING))
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept proc_concept
    ON proc_concept.domain_id = 'Procedure'
   AND proc_concept.source_system = 'allscripts_scm'
   AND proc_concept.source_id = COALESCE(NULLIF(TRIM(ord.IDCode), ''), NULLIF(TRIM(ord.Name), ''), NULLIF(TRIM(oto.TaskName), ''), ord.TypeCode)
  LEFT JOIN _exponent.omop_mapping.domain_source_to_concept mod_concept
    ON mod_concept.domain_id = 'Modifier'
   AND mod_concept.source_system = 'allscripts_scm'
   AND mod_concept.source_id = NULLIF(TRIM(ord.Modifier), '')
  WHERE ord.Active = TRUE
    AND ord.GUID IS NOT NULL
    AND COALESCE(oto.ClientGUID, ord.ClientGUID) IS NOT NULL
    AND COALESCE(oto.PerformedFromDtm, ord.PerformedDtm, ord.SignificantDtm, ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
    AND UPPER(COALESCE(ord.OrderStatusCode, oto.TaskStatusCode, '')) NOT IN ('CAN', 'CANCELLED', 'CANCELED')
    AND UPPER(COALESCE(ord.TypeCode, '')) NOT IN ('MEDICATION', 'MED', 'PHARMACY', 'LAB', 'LABORATORY')
), deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY procedure_occurrence_source_value ORDER BY procedure_datetime DESC) AS rn
  FROM staged
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
FROM deduped
WHERE rn = 1;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.procedure_occurrence AS t
USING silver_procedure_occurrence AS s
ON t.procedure_occurrence_source_value = s.procedure_occurrence_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.procedure_concept_id <=> s.procedure_concept_id)
  OR NOT (t.procedure_date <=> s.procedure_date)
  OR NOT (t.procedure_datetime <=> s.procedure_datetime)
  OR NOT (t.procedure_type_concept_id <=> s.procedure_type_concept_id)
  OR NOT (t.modifier_concept_id <=> s.modifier_concept_id)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.procedure_source_value <=> s.procedure_source_value)
  OR NOT (t.modifier_source_value <=> s.modifier_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id = s.person_id,
  t.procedure_concept_id = s.procedure_concept_id,
  t.procedure_date = s.procedure_date,
  t.procedure_datetime = s.procedure_datetime,
  t.procedure_type_concept_id = s.procedure_type_concept_id,
  t.modifier_concept_id = s.modifier_concept_id,
  t.quantity = s.quantity,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.procedure_source_value = s.procedure_source_value,
  t.procedure_source_concept_id = s.procedure_source_concept_id,
  t.modifier_source_value = s.modifier_source_value,
  t.source_system = s.source_system,
  t.last_mod_tsp = s.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.procedure_occurrence_source_value,
  s.person_id,
  s.procedure_concept_id,
  s.procedure_date,
  s.procedure_datetime,
  s.procedure_type_concept_id,
  s.modifier_concept_id,
  s.quantity,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.procedure_source_value,
  s.procedure_source_concept_id,
  s.modifier_source_value,
  s.source_system,
  s.last_mod_tsp
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_procedure_occurrence (
  source_system,
  procedure_occurrence_source_value,
  active_flag,
  created_tsp,
  last_mod_tsp
)
SELECT
  s.source_system,
  s.procedure_occurrence_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP())
FROM _exponent.omop_silver.procedure_occurrence s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_procedure_occurrence x
  ON s.procedure_occurrence_source_value = x.procedure_occurrence_source_value
WHERE s.source_system = 'allscripts_scm';

In [0]:
%sql
MERGE INTO _exponent.omop_scm.procedure_occurrence AS gold
USING (
  SELECT
    spo.procedure_occurrence_id,
    s.person_id,
    s.procedure_concept_id,
    s.procedure_date,
    s.procedure_datetime,
    s.procedure_type_concept_id,
    s.modifier_concept_id,
    s.quantity,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.procedure_source_value,
    s.procedure_source_concept_id,
    s.modifier_source_value
  FROM _exponent.omop_silver.procedure_occurrence s
  JOIN _exponent.omop_mapping.source_to_procedure_occurrence spo
    ON spo.procedure_occurrence_source_value = s.procedure_occurrence_source_value
   AND spo.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.procedure_occurrence_id = src.procedure_occurrence_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.procedure_concept_id = src.procedure_concept_id,
  gold.procedure_date = src.procedure_date,
  gold.procedure_datetime = src.procedure_datetime,
  gold.procedure_type_concept_id = src.procedure_type_concept_id,
  gold.modifier_concept_id = src.modifier_concept_id,
  gold.quantity = src.quantity,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.procedure_source_value = src.procedure_source_value,
  gold.procedure_source_concept_id = src.procedure_source_concept_id,
  gold.modifier_source_value = src.modifier_source_value

WHEN NOT MATCHED THEN INSERT (
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value
)
VALUES (
  src.procedure_occurrence_id,
  src.person_id,
  src.procedure_concept_id,
  src.procedure_date,
  src.procedure_datetime,
  src.procedure_type_concept_id,
  src.modifier_concept_id,
  src.quantity,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.procedure_source_value,
  src.procedure_source_concept_id,
  src.modifier_source_value
);